In [0]:
dbutils.fs.ls("/Volumes/workspace/default/farewise_data/taxi/")

taxi_df = (
    spark.read
    .option("header", True)
    .csv("/Volumes/workspace/default/farewise_data/taxi/train.csv")
)

print("Total records:", taxi_df.count())

In [0]:
from pyspark.sql import functions as F

taxi_df.select([
    F.count(
        F.when(
            F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1
        )
    ).alias(c)
    for c in taxi_df.columns
]).show(truncate=False)

In [0]:
taxi_typed = (
    taxi_df
    .withColumn("fare_amount", F.col("fare_amount").cast("double"))
    .withColumn("passenger_count", F.col("passenger_count").cast("int"))
    .withColumn("pickup_latitude", F.col("pickup_latitude").cast("double"))
    .withColumn("pickup_longitude", F.col("pickup_longitude").cast("double"))
    .withColumn("dropoff_latitude", F.col("dropoff_latitude").cast("double"))
    .withColumn("dropoff_longitude", F.col("dropoff_longitude").cast("double"))
)

In [0]:
taxi_typed.select(
    F.count(F.when(F.col("fare_amount") <= 0, 1)).alias("invalid_fares"),
    F.count(F.when(
        (F.col("passenger_count") <= 0) |
        (F.col("passenger_count") > 6), 1
    )).alias("invalid_passengers"),
    F.count(F.when(
        F.col("pickup_latitude").isNull() |
        (~F.col("pickup_latitude").between(-90, 90)) |
        (~F.col("pickup_longitude").between(-180, 180)) |
        (~F.col("dropoff_latitude").between(-90, 90)) |
        (~F.col("dropoff_longitude").between(-180, 180)),
        1
    )).alias("invalid_coordinates")
).show()

In [0]:
taxi_typed.select(
    F.min("pickup_datetime").alias("earliest_pickup"),
    F.max("pickup_datetime").alias("latest_pickup")
).show(truncate=False)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-4843884347821363>, line 1
----> 1 taxi_typed.select(
      2     F.min("pickup_datetime").alias("earliest_pickup"),
      3     F.max("pickup_datetime").alias("latest_pickup")
      4 ).show(truncate=False)

NameError: name 'taxi_typed' is not defined

In [0]:

from pyspark.sql import functions as F

taxi_clean = taxi_typed.filter(
    (F.col("fare_amount") > 0) &
    F.col("passenger_count").between(1, 6) &
    F.col("pickup_latitude").between(40, 41) &
    F.col("pickup_longitude").between(-75, -73) &
    F.col("dropoff_latitude").between(40, 41) &
    F.col("dropoff_longitude").between(-75, -73)
)

print("Original records:", taxi_typed.count())
print("Cleaned records:", taxi_clean.count())
print("Removed records:",
      taxi_typed.count() - taxi_clean.count())
